In [ ]:
!poetry install -q

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import datetime
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# OpenMP 다중 로드 허용 및 스레드 경쟁 방지 환경 변수 (최상단 주입 필수)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# 1. 프로젝트 경로 설정 및 환경 변수 명시적 로드
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Docker 네트워크 외부(Host OS)에서 실행되는 Jupyter를 위한 DNS 해석 우회 처리
local_s3_endpoint = os.environ.get("LOCAL_S3_ENDPOINT", "")
if "localstack" in local_s3_endpoint:
    os.environ["LOCAL_S3_ENDPOINT"] = local_s3_endpoint.replace("localstack", "localhost")

In [ ]:
# DataFrame 출력 생략 방지 옵션 설정
pd.set_option('display.max_columns', None)        # 숨김 없이 모든 컬럼 출력
pd.set_option('display.max_colwidth', None)       # 컬럼 안의 긴 텍스트(Dict/List) 전체 출력
pd.set_option('display.expand_frame_repr', False) # 가로 너비 초과 시 줄바꿈 방지
pd.set_option('display.max_rows', 50)             # 필요시 최대 출력 행 수 조정

In [ ]:
import os
import warnings
from pandas.errors import PerformanceWarning

os.environ["LOCAL_S3_ENDPOINT"] = "http://localhost:4566"
os.environ["AWS_ACCESS_KEY_ID"] = "test"
os.environ["AWS_SECRET_ACCESS_KEY"] = "test"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

warnings.filterwarnings("ignore", category=PerformanceWarning)

In [ ]:
# [셀 1] 데이터 파이프라인을 통해 완성된 18종 파생 데이터셋 고속 로드 및 실시간 스트리밍 관제
import time
import boto3
import pandas as pd
from typing import Dict, List, Any
from src.reader.reader_service import ReaderService

# LocalStack S3 메타데이터 스캔을 위한 Boto3 S3 클라이언트 초기화
s3_client = boto3.client(
    "s3",
    endpoint_url="http://localhost:4566",
    aws_access_key_id="test",
    aws_secret_access_key="test",
    region_name="us-east-1"
)

# S3 골드 레이어 최상위 경로 스캔을 통한 파생 실험 버킷 프리픽스 메타데이터 조회
scan_prefix: str = "gold/market_data/gold_daily_asia/"
response_scan = s3_client.list_objects_v2(
    Bucket="data-pipeline-gold",
    Prefix=scan_prefix,
    Delimiter="/"
)

# 스캔된 CommonPrefixes에서 유효한 18종 파생 실험 버킷 ID 목록 추출
bucket_job_ids: List[str] = []
if "CommonPrefixes" in response_scan:
    for prefix_info in response_scan["CommonPrefixes"]:
        folder_name: str = prefix_info["Prefix"].replace(scan_prefix, "").strip("/")
        if folder_name.startswith("bucket_"):
            bucket_job_ids.append(folder_name)

total_buckets_count: int = len(bucket_job_ids)

# reader.yml의 s3_parquet_gold 정책을 조준하는 ReaderService 퍼사드 인스턴스 초기화
reader_service = ReaderService(target_reader="s3_parquet_gold")

# 지연 초기화(Lazy Init) 로그가 배치 루프 출력 중간에 끼어들지 않도록 클라이언트 사전 워밍업 수행
gold_reader = reader_service._get_or_create_reader(source_layer="gold")
if hasattr(gold_reader, "_initialize_client") and getattr(gold_reader, "_client", None) is None:
    gold_reader._client = gold_reader._initialize_client()

# 캡슐화된 ReaderService 퍼사드를 통해 PyArrow C++ Native Bulk 로드 위임
def fetch_dataframe_from_stream(source_path: str, job_id: str) -> pd.DataFrame:
    return reader_service.read_dataframe(
        source_path=source_path,
        job_id=job_id,
        source_layer="gold"
    )

# 18종 통합 데이터셋 레지스트리 및 리포트 요약 레코드 매핑 생성
gold_dataset_repository: Dict[str, pd.DataFrame] = {}
summary_records: List[Dict[str, Any]] = []

print("\n==========================================================================================")
print(f" 🚀 [Total Dataset Ingestion Started] Total Target Datasets: {total_buckets_count}")
print("==========================================================================================")

# 전체 모니터링 타이머 가동
start_total_time: float = time.time()

# 18개 데이터셋 순회 수집, 실시간 스트리밍 관제 사출 및 Asia/Global Outer Join 통합
for index, bucket_job_id in enumerate(bucket_job_ids, start=1):
    bucket_start_time: float = time.time()

    # 버킷 로드 시작 메시지를 개별 라인에 즉시 사출(flush=True)하여 실시간 관제(Observability) 제공
    print(f"[{index:02d}/{total_buckets_count:02d}] {bucket_job_id:<70} ... ", end="", flush=True)

    # PyArrow C++ Native S3FileSystem을 타고 Asia 및 Global 파생 시황 데이터프레임 고속 수집
    asia_dataframe: pd.DataFrame = fetch_dataframe_from_stream(
        source_path=f"gold/market_data/gold_daily_asia/{bucket_job_id}",
        job_id=f"job_asia_{bucket_job_id}"
    )

    global_dataframe: pd.DataFrame = fetch_dataframe_from_stream(
        source_path=f"gold/market_data/gold_daily_global/{bucket_job_id}",
        job_id=f"job_global_{bucket_job_id}"
    )

    # trade_date 기준 Asia/Global 테이블 Outer Join 및 중복 사출 컬럼(_dup) 즉시 정제
    if not asia_dataframe.empty and not global_dataframe.empty:
        combined_dataframe: pd.DataFrame = pd.merge(
            asia_dataframe,
            global_dataframe,
            on="trade_date",
            how="outer",
            suffixes=("", "_dup")
        )
        dup_cols = [col for col in combined_dataframe.columns if col.endswith("_dup")]
        if dup_cols:
            combined_dataframe.drop(columns=dup_cols, inplace=True)
    elif not asia_dataframe.empty:
        combined_dataframe = asia_dataframe
    else:
        combined_dataframe = global_dataframe

    # 거래일 기준 시간순 정렬 및 시계열 분석 전용 DatetimeIndex 바인딩
    if "trade_date" in combined_dataframe.columns:
        combined_dataframe["trade_date"] = pd.to_datetime(combined_dataframe["trade_date"])
        combined_dataframe.sort_values(by="trade_date", ascending=True, inplace=True)
        combined_dataframe.set_index("trade_date", inplace=True)

    # 결합 완성된 데이터셋 저장소 적재 및 단일 버킷 처리 완료 로그 사출
    gold_dataset_repository[bucket_job_id] = combined_dataframe
    elapsed_seconds: float = time.time() - bucket_start_time

    print(f"({elapsed_seconds:.2f}s | Shape: {combined_dataframe.shape})", flush=True)

    # 요약 대시보드 출력을 위한 버킷별 메타데이터 레코드 수집
    summary_records.append({
        "Dataset Bucket ID": bucket_job_id,
        "Total Rows": combined_dataframe.shape[0],
        "Total Features": combined_dataframe.shape[1],
        "Start Date": combined_dataframe.index.min().strftime("%Y-%m-%d") if not combined_dataframe.empty else "N/A",
        "End Date": combined_dataframe.index.max().strftime("%Y-%m-%d") if not combined_dataframe.empty else "N/A",
        "Load Time": f"{elapsed_seconds:.2f}s"
    })

# 전체 배치 타이머 마감 및 정산 대시보드 데이터프레임 구성
total_elapsed: float = time.time() - start_total_time
summary_dataframe: pd.DataFrame = pd.DataFrame(summary_records)

# 최종 수집 정산 대시보드 사출 및 ReaderService 통합 마감 정산
print("\n==========================================================================================")
print(f" 📊 [Total Dataset Ingestion Summary Dashboard] Executed in {total_elapsed:.2f}s")
print("==========================================================================================")
display(summary_dataframe)

reader_service.log_batch_summary()

In [ ]:
display(combined_dataframe.head(5))
display(combined_dataframe.tail(5))

In [ ]:
# [셀 2] 통합 데이터셋에 대한 피처 엔지니어링(타겟 피처 및 파생 피처) 수행
import time
import pandas as pd
from src.feature.feature_service import FeatureService

# 1. 실행 타이머 및 피처 파사드 오케스트레이터 초기화
start_time: float = time.time()
feature_service: FeatureService = FeatureService()

# 2. 통합 정제 데이터프레임(combined_dataframe) 투입 및 피처 엔지니어링 집행
processed_dataframe: pd.DataFrame = feature_service.execute(
    market_data=combined_dataframe
)
elapsed_time: float = time.time() - start_time

# 3. 차원 변화 수치 계산
in_rows, in_cols = combined_dataframe.shape
out_rows, out_cols = processed_dataframe.shape
added_cols = out_cols - in_cols

# 4. 명시적 피처 그룹 매핑 정의
group_column_mappings = {
    "Target Feature": (["target_return_20d"], 1),
    "Trend & Momentum": (["return_lag_5d", "return_lag_20d", "return_lag_60d", "return_lag_120d", "ma_ratio_5_20", "ma_ratio_20_60", "risk_adjusted_return_20d"], 7),
    "Volatility & Risk": (["volatility_20d", "volatility_60d", "vol_regime_ratio", "rolling_skew_20d", "rolling_kurt_20d", "price_position_20d", "norm_atr_20d"], 7),
    "Macro & Cross-Asset": (["us_yield_spread_20d", "korea_us_rate_diff_momentum", "us_kr_market_lag_return", "btc_equity_corr_20d"], 4),
    "Derivatives & Volume": (["proxy_basis_rate", "futures_intraday_range", "volume_anomaly_20d", "value_anomaly_20d"], 4),
    "Calendar & Seasonality": (["month_sin", "month_cos", "is_month_end", "is_quarter_end"], 4)
}

# 5. 그룹별 사출 정산 데이터 구축
processed_cols_set = set(processed_dataframe.columns)
group_breakdown_records = []

for group_name, (expected_cols, target_count) in group_column_mappings.items():
    actual_cols = [col for col in expected_cols if col in processed_cols_set]
    actual_count = len(actual_cols)
    
    group_breakdown_records.append({
        "피처 그룹 (Feature Group)": group_name,
        "활성화 여부": "ENABLED" if actual_count > 0 else "DISABLED",
        "사출 피처 수 (실제 / 목표)": f"{actual_count} / {target_count} 개",
        "생성 피처 목록": ", ".join(actual_cols) if actual_cols else "-"
    })

# 6. 보고서 배너 및 단일 대시보드 표 사출
print("==========================================================================================================================")
print(f"[Feature Engineering Report] Executed in {elapsed_time:.2f}s | Input: {in_rows:,}×{in_cols:,} ➔ Output: {out_rows:,}×{out_cols:,} (+{added_cols} Features)")
print("==========================================================================================================================")
display(pd.DataFrame(group_breakdown_records))

In [ ]:
# [셀 4] 시계열 데이터셋 분할
from src.model.dataset.splitter import DatasetSplitter

# 1. DatasetSplitter 기동 및 시계열 분할 집행
dataset_splitter = DatasetSplitter(
    split_ratios=(0.6, 0.2, 0.2),
    exclude_features=["kis_kospi_daily_close"]
)
split_datasets = dataset_splitter.split(df=processed_dataframe)

# 2. 요약 리포트 프레임 생성
split_summary_report = dataset_splitter.summarize(split_datasets=split_datasets)

# 3. 입력 규격 및 동적 분할 정보(Train/Val/Test 모드 및 비율) 추출
in_rows, in_cols = processed_dataframe.shape
split_mode_name = "Train/Val/Test" if len(dataset_splitter.split_ratios) == 3 else "Train/Test"
ratios_list = ":".join([f"{ratio*100:.0f}" for ratio in dataset_splitter.split_ratios])

print("==========================================================================================================================")
print(f" 📌 [Dataset Splitter Report] Input: ({in_rows:,}, {in_cols:,}) ➔ {split_mode_name} Split to {ratios_list} | Gap: {dataset_splitter.forecast_horizon}d")
print("==========================================================================================================================")
display(split_summary_report)

In [ ]:
# [셀 5] 시계열 데이터셋 피쳐 셀랙션 (Noise & Collinearity Filter ➔ Robust Scaling ➔ 2-Pillar Dynamic Selection)
import copy
from typing import Dict, List, Tuple
import pandas as pd

from src.feature.postprocessor.filter import (
    filter_constant_features,
    filter_missing_ratio,
    filter_zero_ratio,
    pearson_collinearity,
    hierarchical_collinearity
)
from src.feature.postprocessor.scaler import robust_scale
from src.feature.postprocessor.selector import (
    generate_shadow_features,
    elasticnet_importance,
    lightgbm_importance,
    combine_importances,
    select_feature
)

model_ready_datasets = copy.deepcopy(split_datasets)
X_train = model_ready_datasets["X_train"]
y_train = model_ready_datasets["y_train"]

initial_feature_count = X_train.shape[1]
selector_summary: List[Dict[str, str]] = []

# ==========================================================================================================================
# 1. 노이즈 피처 1차 필터링
# ==========================================================================================================================
MAX_MISSING_RATIO = 0.2
MAX_ZERO_RATIO = 0.8
X_train, dropped_constants = filter_constant_features(X_train)
X_train, dropped_missings = filter_missing_ratio(X_train, max_missing_ratio=MAX_MISSING_RATIO)
X_train, dropped_zeros = filter_zero_ratio(X_train, max_zero_ratio=MAX_ZERO_RATIO)

noise_removed_count = len(dropped_constants) + len(dropped_missings) + len(dropped_zeros)
selector_summary.append({
    "단계 (Pipeline Step)": "Noise Filter (Constant & Missing & Zero)",
    "잔여 피처 수 (Features)": f"{X_train.shape[1]:,}",
    "변동 내역 (Changes)": f"-{noise_removed_count} Features (Constant: {len(dropped_constants)}, Missing: {len(dropped_missings)}, Zero: {len(dropped_zeros)})",
    "적용 기준 (Fit Strategy)": f"상수 및 결측률(> {MAX_MISSING_RATIO*100:.0f}%), 0값(≥ {MAX_ZERO_RATIO*100:.0f}%) 기준"
})

# ==========================================================================================================================
# 2. 단변량 및 다변량 공선성 2단계 압축
# ==========================================================================================================================
# 1:1 선형 다공선성 변수 제거
PEARSON_THRESHOLD = 0.85
X_train, dropped_pearson = pearson_collinearity(X_train, threshold=PEARSON_THRESHOLD)
selector_summary.append({
    "단계 (Pipeline Step)": "Pearson Collinearity",
    "잔여 피처 수 (Features)": f"{X_train.shape[1]:,}",
    "변동 내역 (Changes)": f"-{len(dropped_pearson)} Features",
    "적용 기준 (Fit Strategy)": f"피어슨 선형 상관계수(|r| ≥ {PEARSON_THRESHOLD}) 기준"
})

# 상관거리 계층 군집화(HFC)를 통한 다변량 공선성 압축 (타깃 연관도 1위 변수 보존)
DISTANCE_THRESHOLD = 0.40
X_train, dropped_hierarchical = hierarchical_collinearity(
    feature_matrix=X_train,
    target_series=y_train,
    distance_threshold=DISTANCE_THRESHOLD
)
selector_summary.append({
    "단계 (Pipeline Step)": "Hierarchical Feature Clustering (HFC)",
    "잔여 피처 수 (Features)": f"{X_train.shape[1]:,}",
    "변동 내역 (Changes)": f"-{len(dropped_hierarchical)} Features",
    "적용 기준 (Fit Strategy)": f"상관거리 계층 군집화 상관거리 (Distance ≤ {DISTANCE_THRESHOLD})"
})

# ==========================================================================================================================
# 3. X_train 기준 RobustScaler 학습 및 전체 파티션 정규화 (Data Leakage 방지)
# ==========================================================================================================================
current_filtered_features = X_train.columns.tolist()
train_median = X_train.median()

X_train_scaled, _, robust_scaler_instance = robust_scale(X_train=X_train)
model_ready_datasets["X_train"] = X_train_scaled

for partition_key in ["X_val", "X_test", "X_inference"]:
    if partition_key in model_ready_datasets and not model_ready_datasets[partition_key].empty:
        partition_data = model_ready_datasets[partition_key][current_filtered_features]
        partition_data = partition_data.ffill().fillna(train_median[current_filtered_features])
        
        scaled_matrix = robust_scaler_instance.transform(partition_data)
        model_ready_datasets[partition_key] = pd.DataFrame(
            scaled_matrix,
            index=partition_data.index,
            columns=partition_data.columns
        )

selector_summary.append({
    "단계 (Pipeline Step)": "Robust Feature Scaling",
    "잔여 피처 수 (Features)": f"{X_train_scaled.shape[1]:,}",
    "변동 내역 (Changes)": "No Changes (All Partitions Normalized)",
    "적용 기준 (Fit Strategy)": "학습 데이터 중앙값 및 IQR 통계량 기준"
})

# ==========================================================================================================================
# 4. 임베디드 2-Pillar 중요도 산출 및 유계 동적 선별
# ==========================================================================================================================
X_train_shadow, shadow_names = generate_shadow_features(feature_matrix=X_train_scaled)

linear_scores = elasticnet_importance(
    scaled_feature_matrix=X_train_shadow, 
    target_series=y_train
)

tree_scores = lightgbm_importance(
    feature_matrix=X_train_shadow,
    target_series=y_train,
)

combined_feature_scores = combine_importances(
    linear_importance=linear_scores,
    tree_importance=tree_scores,
    shadow_feature_names=shadow_names,
)

TARGET_CUMULATIVE_THRESHOLD = 0.95
MIN_FEATURES_BOUND = 10
MAX_FEATURES_BOUND = 50

selected_features, achieved_cumulative_ratio = select_feature(
    feature_scores=combined_feature_scores,
    cumulative_threshold=TARGET_CUMULATIVE_THRESHOLD,
    min_features=MIN_FEATURES_BOUND,
    max_features=MAX_FEATURES_BOUND
)

for partition_key in ["X_train", "X_val", "X_test", "X_inference"]:
    if partition_key in model_ready_datasets and not model_ready_datasets[partition_key].empty:
        model_ready_datasets[partition_key] = model_ready_datasets[partition_key][selected_features]

selector_summary.append({
    "단계 (Pipeline Step)": f"Dynamic Feature Selection (Top {len(selected_features)})",
    "잔여 피처 수 (Features)": f"{len(selected_features):,}",
    "변동 내역 (Changes)": f"Selected {len(selected_features)} Features",
    "적용 기준 (Fit Strategy)": f"ElasticNet + LightGBM (Target: {int(TARGET_CUMULATIVE_THRESHOLD * 100)}%)"
})

# ==========================================================================================================================
# 5. 정산 대시보드 리포트 생성 및 사출
# ==========================================================================================================================
selector_summary_report = pd.DataFrame(selector_summary).set_index("단계 (Pipeline Step)")

in_rows, in_cols = split_datasets["X_train"].shape
out_rows, out_cols = model_ready_datasets["X_train"].shape

print("=" * 122)
print(f" 📌 [Feature Selector Report] Input: ({in_rows:,}, {initial_feature_count:,}) ➔ Output: ({out_rows:,}, {out_cols:,}) | Selected: {len(selected_features)} Features (Coverage: {achieved_cumulative_ratio * 100:.1f}%)")
print("=" * 122)
display(selector_summary_report)